# Create Test Set (imagesTs / labelsTs)

Randomly sample matching frame/mask pairs from `output/frames` and `output/masks`
(produced by `sam2_segmentation.ipynb`) and copy them into new `imagesTs` / `labelsTs`
folders using nnU-Net naming conventions.

- Source folders are left untouched.
- `imagesTs/img_NNN_0000.png` (converted from JPG to PNG)
- `labelsTs/img_NNN.png` (copied as-is)

## 1. Configuration

In [ ]:
import os
import random
import shutil

from PIL import Image

BASE_DIR = r"\\wsl.localhost\Ubuntu\home\mci\BHS_project\sam2-surgical-segmentation"
FRAMES_DIR = os.path.join(BASE_DIR, "output", "frames")
MASKS_DIR = os.path.join(BASE_DIR, "output", "masks")

IMAGES_TS_DIR = os.path.join(BASE_DIR, "imagesTs")
LABELS_TS_DIR = os.path.join(BASE_DIR, "labelsTs")

NUM_SAMPLES = 200  # how many frame/mask pairs to randomly select
SEED = 42

os.makedirs(IMAGES_TS_DIR, exist_ok=True)
os.makedirs(LABELS_TS_DIR, exist_ok=True)

print("Frames dir:", FRAMES_DIR)
print("Masks dir:", MASKS_DIR)
print("imagesTs ->", IMAGES_TS_DIR)
print("labelsTs ->", LABELS_TS_DIR)

## 2. Find matching frame/mask pairs

In [ ]:
frame_indices = {
    f.split("_")[1].split(".")[0]
    for f in os.listdir(FRAMES_DIR)
    if f.startswith("frame_") and f.endswith(".jpg")
}
mask_indices = {
    f.split("_")[1].split(".")[0]
    for f in os.listdir(MASKS_DIR)
    if f.startswith("frame_") and f.endswith(".png")
}

common_indices = sorted(frame_indices & mask_indices)
print(f"Found {len(common_indices)} matching frame/mask pairs")

## 3. Randomly select pairs and copy to imagesTs / labelsTs

In [ ]:
random.seed(SEED)
num_to_select = min(NUM_SAMPLES, len(common_indices))
selected = sorted(random.sample(common_indices, num_to_select))

for new_idx, frame_idx in enumerate(selected, start=1):
    frame_path = os.path.join(FRAMES_DIR, f"frame_{frame_idx}.jpg")
    mask_path = os.path.join(MASKS_DIR, f"frame_{frame_idx}.png")

    img = Image.open(frame_path).convert("RGB")
    img.save(os.path.join(IMAGES_TS_DIR, f"img_{new_idx:03d}_0000.png"))

    shutil.copy(mask_path, os.path.join(LABELS_TS_DIR, f"img_{new_idx:03d}.png"))

print(f"Selected {len(selected)} pairs")
print(f"Images saved to: {IMAGES_TS_DIR}")
print(f"Labels saved to: {LABELS_TS_DIR}")

## 4. Preview a sample pair

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

sample_idx = 1
img = Image.open(os.path.join(IMAGES_TS_DIR, f"img_{sample_idx:03d}_0000.png"))
mask = np.array(Image.open(os.path.join(LABELS_TS_DIR, f"img_{sample_idx:03d}.png")))

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(img)
axes[0].set_title(f"img_{sample_idx:03d}_0000.png")
axes[0].axis("off")

axes[1].imshow(mask, cmap="viridis", vmin=0, vmax=max(mask.max(), 1))
axes[1].set_title(f"img_{sample_idx:03d}.png (unique values: {np.unique(mask)})")
axes[1].axis("off")

plt.tight_layout()
plt.show()